In [ ]:
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig
from typing import Any


@before_model
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """
    在模型处理之前修剪消息，只保留最近的几条消息以适应上下文窗口限制。
    
    修剪策略：
    - 如果消息总数 <= 3，不进行处理
    - 否则：保留第 1 条消息（通常是系统消息或首条消息）+ 最近 3-4 条消息
    - 这样可以保留重要的初始上下文，同时控制 token 数量
    """
    messages = state["messages"]

    # 如果消息数量不多，无需修剪
    if len(messages) <= 3:
        return None  # 返回 None 表示不修改状态

    # 第一条消息通常是系统消息或对话起点，需要保留
    first_msg = messages[0]
    
    # 根据消息总数的奇偶性决定保留最近 3 条还是 4 条消息
    # 这样可以保持 Human-AI 对话对的完整性（避免只保留单方面的消息）
    recent_messages = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]
    
    # 组合新消息列表：第一条 + 最近的消息
    new_messages = [first_msg] + recent_messages

    # 返回更新指令：
    # 1. RemoveMessage(id=REMOVE_ALL_MESSAGES) - 清除所有现有消息
    # 2. *new_messages - 添加修剪后的消息
    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_messages
        ]
    }


# 创建 Agent，将 trim_messages 中间件注册到 middleware 列表中
agent = create_agent(
    model,  # 使用之前定义的模型
    tools=[],  # 这里可以添加自定义工具
    middleware=[trim_messages],  # 注册消息修剪中间件
    checkpointer=InMemorySaver(),  # 启用短期记忆持久化
)

# 配置线程 ID，同一个 thread_id 的对话会共享记忆
config: RunnableConfig = {"configurable": {"thread_id": "1"}}

In [ ]:
# 模拟多轮对话，演示消息修剪效果

# 第一次调用：用户自我介绍
print("=== 第 1 轮：用户自我介绍 ===")
result1 = agent.invoke({"messages": "hi, my name is bob"}, config)
print(f"AI 回复：{result1['messages'][-1].content[:100]}...\n")

# 第二次调用：请求写诗
print("=== 第 2 轮：请求写关于猫的诗 ===")
result2 = agent.invoke({"messages": "write a short poem about cats"}, config)
print(f"AI 回复：{result2['messages'][-1].content[:100]}...\n")

# 第三次调用：继续请求
print("=== 第 3 轮：请求写关于狗的诗 ===")
result3 = agent.invoke({"messages": "now do the same but for dogs"}, config)
print(f"AI 回复：{result3['messages'][-1].content[:100]}...\n")

# 第四次调用：测试记忆 —— 此时早期的消息已被修剪，但 AI 仍应记得用户的名字
print("=== 第 4 轮：测试记忆（询问用户名字）===")
final_response = agent.invoke({"messages": "what's my name?"}, config)
final_response["messages"][-1].pretty_print()

### 消息修剪过程详解


**修剪策略可视化：**

```
初始消息列表（假设 7 条消息）:
[Msg0, Msg1, Msg2, Msg3, Msg4, Msg5, Msg6]
 │                     └───────┬───────┘
 │                             └─ 最近 3 条 (保留)
 └─ 第一条 (保留)

修剪后结果:
[Msg0, Msg4, Msg5, Msg6]
```

**关键点说明：**

1. **`@before_model` 装饰器**: 
   - 让 `trim_messages` 函数在模型处理之前执行
   - 每次 Agent 被调用时都会运行此中间件

2. **`RemoveMessage(id=REMOVE_ALL_MESSAGES)`**:
   - 这是一个特殊指令，告诉 LangGraph 清除状态中的所有消息
   - 然后重新添加修剪后的消息列表

3. **为什么保留第一条消息？**:
   - 第一条消息通常包含重要的初始上下文（如系统提示、用户自我介绍）
   - 保留它可以让 AI 记住对话的起点

4. **为什么根据奇偶性调整保留数量？**:
   - Human-AI 对话是成对的
   - 奇偶判断确保不会只保留单方面的消息（如只有 Human 消息而没有 AI 回复）

**预期输出：**
```
================================== Ai Message ==================================

Your name is Bob. You told me that earlier.
If you'd like me to call you a nickname or use a different name, just say the word.
```

即使中间的消息被修剪了，AI 仍然能记住用户的名字，因为第一条消息（包含自我介绍）被保留了。

# Usage 用法
通过在创建Agent的时候指定 **checkpointer** 来添加 **线程级别的持久化记忆**

【tips】

- Langchain的Agent将短期记忆作为Agent state的一部分进行管理
- 通过将这些信息存储在图的状态中，Agent可以在不同的线程中保持分离的同时，访问给定的完整上下文
- state通过checkpointer持久化到数据库（或者内存）中，以便线程能随时恢复
- 短期记忆在Agent被调用或一个步骤（如工具调用）完成时更新，并且在每个步骤开始时读取状态

In [1]:
from langchain_openai import ChatOpenAI
import os

model = ChatOpenAI(
    model="deepseek-ai/DeepSeek-V3.2",
    #! 这里要注意，模型的Tools能力也挂钩
    base_url="https://api.siliconflow.cn/v1",
    api_key=os.environ.get("SILICONFLOW_API_KEY"),
    temperature=0.2,
)


In [9]:
from langchain.tools import tool, ToolRuntime
from langgraph.store.memory import InMemoryStore

# 模拟用户数据库
USER_DATABASE = {
    "user_123": {
        "name": "John Smith",
        "email": "john@example.com",
        "balance": 5000
    },
    "user_456": {
        "name": "Jane Doe", 
        "email": "jane@example.com",
        "balance": 3000
    }
}

@tool
def get_user_info(runtime: ToolRuntime) -> str:
    """查询用户信息。从 Store 中获取 user_id，然后查询用户数据库返回详细信息。"""
    store = runtime.store
    
    # 从 store 中获取 user_id (使用 thread_id 作为 key)
    user_id_item: dict = store.get(("session",), "user_id")
    
    if not user_id_item:
        return "未找到用户 ID，请先登录"
    
    user_id = user_id_item.value
    
    # 查询数据库
    user = USER_DATABASE.get(user_id)
    
    if user:
        return f"用户信息：\n姓名：{user['name']}\n邮箱：{user['email']}\n余额：{user['balance']}"
    else:
        return f"未找到用户 ID: {user_id}"


@tool
def login(user_id: str, runtime: ToolRuntime) -> str:
    """用户登录，将 user_id 保存到 Store 中"""
    store = runtime.store
    
    # 将 user_id 保存到 store，使用 thread_id 作为 key
    store.put(("session",), "user_id", user_id) # session作为namespace
    
    return f"登录成功！用户 ID: {user_id} 已保存"

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

# Saver是存储对话历史的
# Store是存储结构化信息的

# 创建 Store 实例用于持久化用户 ID
store = InMemoryStore()

agent = create_agent(
    model = model, 
    tools=[get_user_info, login],
    checkpointer=InMemorySaver(),
    store=store,  # 绑定 store 到 agent
    system_prompt="多用工具查询，如果用户没有登录，引导用户先登录"
)


# 第一次调用：先登录，保存 user_id
print("=== 第一次调用：登录 ===")
result = agent.invoke(
    {"messages": [{"role": "user", 
                   "content": "登录 user_123"
                   }]}, 
    {"configurable": {"thread_id": "1"}}
)
print(result["messages"][-1].content)

# 第二次调用：查询信息（无需再传 user_id，因为已经保存在 Store 中）
print("\n=== 第二次调用：查询信息 ===")
result = agent.invoke(    
    {"messages": [{"role": "user", "content": "我的信息是啥"}]},   
    {"configurable": {"thread_id": "1"}}
)

print(result["messages"][-1].content)

# 第三次调用：换一个 thread_id，应该查询不到（因为不同会话）
print("\n=== 第三次调用：新会话 (thread_id=2) ===")
result = agent.invoke(    
    {"messages": [{"role": "user", "content": "查询我信息"}]},   
    {"configurable": {"thread_id": "2"}}
)

print(result["messages"][-1].content)

=== 第一次调用：登录 ===
登录成功！用户 user_123 已成功登录。现在您可以查询用户信息了。

=== 第二次调用：查询信息 ===
您的用户信息如下：

**姓名：** John Smith  
**邮箱：** john@example.com  
**余额：** 5000

您已成功登录为 user_123，账户信息如上所示。

=== 第三次调用：新会话 (thread_id=2) ===
目前我无法直接查询您的用户信息。要查询用户信息，需要先进行登录操作。

请您先使用登录功能，提供您的用户ID，这样我就能帮您查询详细信息了。

您可以使用以下格式登录：
```
登录 [您的用户ID]
```

或者直接告诉我您的用户ID，我来帮您完成登录流程。


### 【InMemorySaver & InMemoryStore 区别】

`InMemorySaver` 和 `InMemoryStore` 虽然都是存储在内存中的组件，但它们服务的**记忆类型**和**数据组织方式**有本质区别。

以下是两者的详细对比分析：

1. InMemorySaver (即 Checkpointer) —— 短期记忆
`InMemorySaver` 是一个 **检查点（Checkpointer）** 的内存实现，主要用于管理 **短期记忆（Short-term memory）** 。

*   **核心功能**：实现**线程级持久化（Thread-level persistence）**。它让 Agent 能够记住单个对话线程内的交互信息。
*   **组织单位**：通过 **`thread_id`** 来隔离数据。每个线程都有自己独立的状态快照。
*   **存储内容**：存储的是整个 `State`（状态），通常包括对话历史（messages）和自定义的状态字段。
*   **工作机制**：当 Agent 运行完一个步骤（如工具调用）时，`InMemorySaver` 会保存当前的快照；在下一次交互开始时，它会根据 `thread_id` 加载之前的快照以恢复上下文。
*   **局限性**：由于是“InMemory”，一旦程序重启，所有的短期记忆都会丢失。在生产环境中，通常会被 `SqliteSaver` 或 `PostgresSaver` 取代。

2. InMemoryStore —— 长期记忆
虽然来源中没有详细展开 `InMemoryStore` 的代码实现，但在 LangChain 的体系中（如前文对话所述），它是 **长期记忆（Long-term memory）** 的内存实现。

*   **核心功能**：实现 **跨线程持久化** 。它存储的信息不局限于某一个对话，而是可以被同一个用户的所有对话、甚至所有用户共享。
*   **组织单位**：使用 **命名空间/键模式（Namespace/Key pattern）** 。例如，你可以使用 `("user_123", "preferences")` 这样的路径来存储数据。
*   **存储内容**：存储的是结构化的知识或偏好信息（如“用户喜欢简洁的回答”），而不是原始的对话序列。
*   **工作机制**：在工具执行过程中，可以通过 `runtime.store` 显式地进行 `put`（写入）和 `get`（读取）操作。
*   **应用场景**：用于存储那些需要跨会话保留的背景知识、用户设置或历史总结。

3. 核心差异总结表

| 特性 | InMemorySaver (Checkpointer) | InMemoryStore (Store) |
| :--- | :--- | :--- |
| **记忆类型** | **短期记忆** (Thread-local) | **长期记忆** (Cross-thread) |
| **主要用途** | 维持当前对话的连贯性 | 存储跨会话的知识或用户偏好 |
| **检索索引** | `thread_id` | `Namespace` + `Key` |
| **自动程度** | **自动**。每步运行后由框架自动保存快照 | **手动**。通常需要在工具中显式调用读写 |
| **典型数据** | `[HumanMessage, AIMessage, ...]` | `{"language": "Chinese", "style": "brief"}` |
| **生产替代品** | `PostgresSaver`, `SqliteSaver` | `PostgresStore` |

**简单一句话总结**：
如果你想让 AI 记得 **“刚才我们聊了什么”**，请使用 `InMemorySaver`；

如果你想让 AI 记得 **“这个用户是谁，他有什么长期的喜好”** ，请使用 `InMemoryStore`。

# Customizing Agent Memory 
自定义Agent记忆

也就是添加一些自定义字段。

默认情况下，代理使用 **AgentState** 来管理短期记忆，具体通过 **messages** 键来管理 **对话历史**

可以通过扩展 AgentState 来扩展额外的字段。通过 state_schema 参数传递给 create_agent

In [ ]:
from langchain.agents import AgentState, create_agent
from langgraph.checkpoint.memory import InMemorySaver
# from langgraph.store.memory import InMemoryStore
from langchain.messages import HumanMessage
import json

#? class AgentState(TypedDict, Generic[ResponseT]):
# AgentState 是 TypedDict 的子类，可以直接定义属性

class CustomAgentState(AgentState):
    user_id: str
    preferences: dict  # 用户偏好 dict
    
agent = create_agent(
    model,
    tools=[],
    state_schema=CustomAgentState,
    checkpointer=InMemorySaver(),
    # context_schema=
)
# input: _InputAgentState | Command | None,
# 
res = agent.invoke(
    {
        "messages": [("human", "我的 user_id 是多少")],
        "user_id": "user_chasen",
        "preferences": {"theme": "dark"}
    },
    {"configurable": {"thread_id": "1"}}
)

# 美化输出
output = {
    "messages": [
        {
            "type": msg.__class__.__name__,
            "content": msg.content[:150] + "..." if len(msg.content) > 150 else msg.content
        }
        for msg in res["messages"]
    ], # 这一整个是一个列表推导式
    "user_id": res.get("user_id"),
    "preferences": res.get("preferences")
}

print(json.dumps(output, indent=2, ensure_ascii=False))

{
  "messages": [
    {
      "type": "HumanMessage",
      "content": "我的 user_id 是多少"
    },
    {
      "type": "AIMessage",
      "content": "我无法直接获取您的用户ID信息哦！😊\n\n用户ID通常是在您注册或登录某个平台时由系统分配的，具体取决于您当前使用的平台或应用。比如：\n\n- 如果您在问某个特定网站或APP的用户ID，可以查看个人资料页或账户设置\n- 如果是微信、QQ等社交平台，可以在个人资料中找到\n- 如果是某个游戏或软件，通常在账..."
    }
  ],
  "user_id": "user_chasen",
  "preferences": {
    "theme": "dark"
  }
}


简单说 **state_schema** 只是定义了 state 的结构，但你要自己负责把这些字段注入到 prompt 或 messages 中，Agent 才能"看到"并使用它们；或者这些字段也可以用于控制应用程序

# 常见模式
启动了短期记忆后，由于会存储聊天历史，这会导致长对话可能会超过LLM的上下文窗口。参加的解决方案有：

- Trim Messages：修剪信息，移除一些信息，比如说最早的5条
- Delete Messages：永久删除 langgraph 状态中的消息
- Summarize Messages：总结消息，总结早期消息，并且用摘要替换他们
- Custom Messages：自定义策略，比如说消息过滤

In [ ]:
from langchain.messages import AnyMessage, RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig
from typing import Any


@before_model
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """
    在模型处理之前修剪消息，只保留最近的几条消息以适应上下文窗口限制。
    
    修剪策略：
    - 如果消息总数 <= 3，不进行处理
    - 否则：保留第 1 条消息（通常是系统消息或首条消息）+ 最近 3-4 条消息
    - 这样可以保留重要的初始上下文，同时控制 token 数量
    """
    messages = state["messages"]

    # 如果消息数量不多，无需修剪
    if len(messages) <= 3:
        return None  # 返回 None 表示不修改状态

    # 第一条消息通常是系统消息或对话起点，需要保留
    first_msg = messages[0]
    
    # 根据消息总数的奇偶性决定保留最近 3 条还是 4 条消息
    # 这样可以保持 Human-AI 对话对的完整性（避免只保留单方面的消息）
    recent_messages: list[AnyMessage] = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]
    
    # 组合新消息列表：第一条 + 最近的消息
    new_messages = [first_msg] + recent_messages

    # 返回更新指令：
    # 1. RemoveMessage(id=REMOVE_ALL_MESSAGES) - 清除所有现有消息
    # 2. *new_messages - 添加修剪后的消息
    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_messages # 解包 list[AnyMessage]
        ]
    }
    
# 创建 Agent，将 trim_messages 中间件注册到 middleware 列表中
agent = create_agent(
    model,  # 使用之前定义的模型
    tools=[],  # 这里可以添加自定义工具
    middleware=[trim_messages],  # 注册消息修剪中间件
    checkpointer=InMemorySaver(),  # 启用短期记忆持久化
)

# 配置线程 ID，同一个 thread_id 的对话会共享记忆
config: RunnableConfig = {"configurable": {"thread_id": "1"}}

# 模拟多轮对话，演示消息修剪效果

# 第一次调用：用户自我介绍
print("=== 第 1 轮：用户自我介绍 ===")
result1 = agent.invoke({"messages": "hi, my name is bob"}, config)
print(f"AI 回复：{result1['messages'][-1].content[:100]}...\n")

# 第二次调用：请求写诗
print("=== 第 2 轮：请求写关于猫的诗 ===")
result2 = agent.invoke({"messages": "write a short poem about cats"}, config)
print(f"AI 回复：{result2['messages'][-1].content[:100]}...\n")

# 第三次调用：继续请求
print("=== 第 3 轮：请求写关于狗的诗 ===")
result3 = agent.invoke({"messages": "now do the same but for dogs"}, config)
print(f"AI 回复：{result3['messages'][-1].content[:100]}...\n")

# 第四次调用：测试记忆 —— 此时早期的消息已被修剪，但 AI 仍应记得用户的名字
print("=== 第 4 轮：测试记忆（询问用户名字）===")
final_response = agent.invoke({"messages": "what's my name?"}, config)
final_response["messages"][-1].pretty_print()

=== 第 1 轮：用户自我介绍 ===
AI 回复：Hi Bob! Nice to meet you. 😊  
Is there anything I can help you with today?...

=== 第 2 轮：请求写关于猫的诗 ===
AI 回复：Of course, Bob! Here's a short poem about our feline friends.

---

Soft paws on the floor,
A purr l...

=== 第 3 轮：请求写关于狗的诗 ===
AI 回复：Of course, Bob! Here's a poem for our loyal canine companions.

---

A thumping tail on the floor,
A...

=== 第 4 轮：测试记忆（询问用户名字）===
================================== Ai Message ==================================

Your name is **Bob**! It's nice to meet you. 😊
